# Cold-Start Analysis

**Research question:** How does model accuracy degrade for articles with little or no sales history?

In fashion retail, 15.3% of articles have ≤ 3 weeks of training history (new product launches,
seasonal introductions). The lag features (`sales_lag1`, `sales_rolling4_mean`) that drive
97.8% of the SHAP signal carry almost no information for these articles.

**Methodology:**
1. Compute each article's `product_age_weeks` at the START of the test period (2020-01-06)
2. Classify into three tiers:

| Tier | Age at test start | Interpretation |
|------|------------------|----------------|
| **Cold** | ≤ 4 weeks | Just launched — effectively no lag history |
| **Warm** | 5 – 16 weeks | One quarter of history |
| **Established** | > 16 weeks | Full seasonal cycle seen |

3. Evaluate all models on each tier separately
4. Quantify how much the ML advantage shrinks for cold-start articles


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 130
os.makedirs('plots/cold_start', exist_ok=True)

BASE = '/Users/kamilaya/Desktop/thesis'

def rmse(a, b):
    return np.sqrt(mean_squared_error(np.asarray(a), np.clip(np.asarray(b), 0, None)))

def nz_mape(a, b):
    a, b = np.asarray(a), np.clip(np.asarray(b), 0, None)
    mask = a > 0
    return np.mean(np.abs((a[mask]-b[mask])/a[mask]))*100 if mask.sum() else np.nan

def score_seg(label, y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return {
        'model': label,
        'RMSE':         round(rmse(y_true, y_pred), 4),
        'MAE':          round(mean_absolute_error(y_true, y_pred), 4),
        'NZ-MAPE (%)':  round(nz_mape(y_true, y_pred), 2),
        'n_rows':       int(len(y_true)),
    }


In [2]:
# Load full preprocessed data (unscaled) and test split (scaled for ML models)
df     = pd.read_csv(f'{BASE}/tshirts_preprocessed.csv', parse_dates=['week_start'])
test   = pd.read_csv(f'{BASE}/tshirts_test.csv',         parse_dates=['week_start'])

TEST_START = pd.Timestamp('2020-01-06')
TRAIN_END  = pd.Timestamp('2019-04-30')
VAL_END    = pd.Timestamp('2019-12-31')

# Compute article age as weeks since FIRST NON-ZERO SALE before the test period.
# Using raw week count would give every article 68 weeks (full-grid expansion artifact).
history = df[df['week_start'] < TEST_START].copy()
history['sales_raw'] = np.expm1(history['log_sales_volume'])

first_sale = (
    history[history['sales_raw'] > 0]
    .groupby('article_id')['week_start']
    .min()
    .rename('first_sale_week')
    .reset_index()
)

all_articles = pd.DataFrame({'article_id': df['article_id'].unique()})
article_age = all_articles.merge(first_sale, on='article_id', how='left')
article_age['age_at_test_start'] = (
    (TEST_START - article_age['first_sale_week']).dt.days // 7
).clip(lower=0).fillna(0).astype(int)

print(f'Articles with any non-zero sales before test: {article_age["age_at_test_start"].gt(0).sum():,}')
print(f'Articles with zero sales history (cold):     {article_age["age_at_test_start"].eq(0).sum():,}')
print(f'\nAge distribution (weeks since first sale):')
print(article_age['age_at_test_start'].describe().round(1))

# Pre-join naive predictions (sales_lag1 from unscaled df) into test by (article_id, week_start).
# This guarantees row-level alignment — no truncation or re-sorting needed later.
_naive_src = df[df['week_start'] >= TEST_START][['article_id', 'week_start', 'sales_lag1']].copy()
_naive_src['naive_pred_raw'] = np.expm1(_naive_src['sales_lag1'].fillna(0)).clip(0, None)
test = test.merge(
    _naive_src[['article_id', 'week_start', 'naive_pred_raw']],
    on=['article_id', 'week_start'], how='left'
)
test['naive_pred_raw'] = test['naive_pred_raw'].fillna(0)
print(f'\nNaive pred merged — test shape: {test.shape}')


Articles with any non-zero sales before test: 5,812
Articles with zero sales history (cold):     2,061

Age distribution (weeks since first sale):
count    7873.0
mean       37.9
std        27.5
min         0.0
25%         0.0
50%        43.0
75%        67.0
max        68.0
Name: age_at_test_start, dtype: float64

Naive pred merged — test shape: (299174, 114)


In [3]:
# Classify into cold / warm / established (article_age already computed in cs_load)
def assign_tier(age):
    if age <= 4:
        return 'Cold (≤4w)'
    elif age <= 16:
        return 'Warm (5–16w)'
    else:
        return 'Established (>16w)'

article_age['tier'] = article_age['age_at_test_start'].apply(assign_tier)
tier_counts = article_age['tier'].value_counts()
print('Tier distribution:')
for t in ['Cold (≤4w)', 'Warm (5–16w)', 'Established (>16w)']:
    n = tier_counts.get(t, 0)
    pct = n / len(article_age) * 100
    print(f'  {t:<22} {n:>5,} articles  ({pct:.1f}%)')

# Merge tier into test set
test = test.merge(article_age[['article_id', 'age_at_test_start', 'tier']], on='article_id', how='left')
test['tier'] = test['tier'].fillna('Cold (≤4w)')   # unseen articles are fully cold
test['y_raw'] = np.expm1(test['log_sales_volume'])

print(f'\nTest rows by tier:')
print(test.groupby('tier')['y_raw'].agg(['count', 'mean']).round(2).rename(
    columns={'count': 'n_rows', 'mean': 'mean_sales'}))


Tier distribution:
  Cold (≤4w)             2,146 articles  (27.3%)
  Warm (5–16w)             252 articles  (3.2%)
  Established (>16w)     5,475 articles  (69.5%)

Test rows by tier:
                    n_rows  mean_sales
tier                                  
Cold (≤4w)           81548        5.57
Established (>16w)  208050        1.16
Warm (5–16w)          9576        4.49


In [4]:
# Load all 6 saved ML models + replicate Naïve and lag-based baselines on test set

OHE_PREFIXES = ('index_group_name_', 'colour_group_name_',
                'graphical_appearance_name_', 'perceived_colour_value_name_')

def normalise(df_in):
    return df_in.rename(columns={c: c.replace(' ', '_')
                                  for c in df_in.columns if any(c.startswith(p) for p in OHE_PREFIXES)})

test_orig = test.copy()
test_norm = normalise(test.copy())

model_files = {
    'LightGBM (baseline)':     'LightGBM_Baselin.pkl',
    'XGBoost (baseline)':      'XGBoost_Baselin.pkl',
    'RandomForest (baseline)': 'RandomForest_Baselin.pkl',
}

loaded_models = {}
for label, fname in model_files.items():
    fpath = f'{BASE}/saved_models/{fname}'
    if os.path.exists(fpath):
        loaded_models[label] = joblib.load(fpath)
        print(f'Loaded: {label}')
    else:
        print(f'[SKIP] {label} not found at {fpath}')

import os


Loaded: LightGBM (baseline)
Loaded: XGBoost (baseline)
Loaded: RandomForest (baseline)


In [5]:
TIERS   = ['Cold (≤4w)', 'Warm (5–16w)', 'Established (>16w)']
results = []

for tier in TIERS:
    mask = test['tier'] == tier
    sub  = test[mask].copy()
    if len(sub) == 0:
        continue
    y_true = sub['y_raw'].values

    # ── Naïve baseline ─────────────────────────────────────────────────────
    # naive_pred_raw was pre-joined in cs_load by (article_id, week_start),
    # so it is perfectly aligned with y_true — no truncation needed.
    naive_pred = sub['naive_pred_raw'].values
    r = score_seg('Naïve', y_true, naive_pred)
    r['tier'] = tier; results.append(r)

    # ── ML models ─────────────────────────────────────────────────────────
    for label, model in loaded_models.items():
        if hasattr(model, 'feature_name_'):
            model_feats = list(model.feature_name_)
            X = test_norm
        else:
            model_feats = [str(f) for f in model.feature_names_in_]
            X = test_orig
        feats = [f for f in model_feats if f in X.columns]
        X_sub = X[mask][feats]
        pred  = np.expm1(np.clip(model.predict(X_sub), 0, None))
        r = score_seg(label, y_true, pred)
        r['tier'] = tier; results.append(r)

results_df = pd.DataFrame(results)
print('Cold-start evaluation complete.')
print()
for tier in TIERS:
    sub = results_df[results_df['tier'] == tier].sort_values('RMSE')
    if len(sub) == 0:
        print(f'  {tier}: no articles in this tier — skipped.')
        continue
    n_rows = sub['n_rows'].values[0]
    print(f'{"="*55}')
    print(f'  {tier}  (n_rows={n_rows:,})')
    print(f'{"="*55}')
    print(sub[['model', 'RMSE', 'MAE', 'NZ-MAPE (%)']].to_string(index=False))
    print()


Cold-start evaluation complete.

  Cold (≤4w)  (n_rows=81,548)
                  model         RMSE          MAE  NZ-MAPE (%)
    LightGBM (baseline) 1.001620e+01 2.535800e+00 7.527000e+01
RandomForest (baseline) 1.007130e+01 2.597600e+00 8.343000e+01
     XGBoost (baseline) 1.012540e+01 2.569900e+00 7.569000e+01
                  Naïve 1.145681e+70 7.883059e+68 5.292980e+69

  Warm (5–16w)  (n_rows=9,576)
                  model         RMSE          MAE  NZ-MAPE (%)
RandomForest (baseline) 6.762000e+00 1.732500e+00 6.772000e+01
    LightGBM (baseline) 6.837100e+00 1.710500e+00 6.203000e+01
     XGBoost (baseline) 6.981600e+00 1.754800e+00 6.278000e+01
                  Naïve 1.100820e+70 7.352739e+68 2.071873e+69

  Established (>16w)  (n_rows=208,050)
                  model         RMSE          MAE  NZ-MAPE (%)
RandomForest (baseline) 3.742900e+00 4.711000e-01 6.444000e+01
    LightGBM (baseline) 3.818600e+00 4.662000e-01 5.966000e+01
     XGBoost (baseline) 3.941300e+00 4.762000e

In [6]:
# ── Plot 1: RMSE by tier and model ────────────────────────────────────────────
pivot = results_df.pivot_table(index='tier', columns='model', values='RMSE')
pivot = pivot.reindex(['Cold (≤4w)', 'Warm (5–16w)', 'Established (>16w)'])

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(pivot))
width = 0.18
colors  = ['#c0392b', '#2980b9', '#27ae60', '#8e44ad']
models_ordered = ['Naïve'] + list(loaded_models.keys())

for i, model in enumerate(models_ordered):
    if model not in pivot.columns: continue
    vals = pivot[model].values
    offset = (i - len(models_ordered)/2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=model, color=colors[i], alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.04,
                f'{v:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(pivot.index, fontsize=11)
ax.set_ylabel('Test RMSE (original scale)')
ax.set_title('Model RMSE by Article Age Tier\n(Cold-Start Analysis)', fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('plots/cold_start/CS1_rmse_by_tier.png')
plt.close()

# ── Plot 2: ML advantage (Δ RMSE vs Naïve) by tier ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ml_models = list(loaded_models.keys())

tier_labels = ['Cold (≤4w)', 'Warm (5–16w)', 'Established (>16w)']
x = np.arange(len(tier_labels))
width = 0.22

for i, model in enumerate(ml_models):
    deltas = []
    for tier in tier_labels:
        tier_sub  = results_df[results_df['tier'] == tier]
        naive_r   = tier_sub[tier_sub['model'] == 'Naïve']['RMSE'].values
        model_r   = tier_sub[tier_sub['model'] == model]['RMSE'].values
        if len(naive_r) and len(model_r):
            deltas.append((naive_r[0] - model_r[0]) / naive_r[0] * 100)
        else:
            deltas.append(0)
    offset = (i - 1) * width
    bars = ax.bar(x + offset, deltas, width, label=model,
                  color=colors[i+1], alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, deltas):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+(0.3 if v>=0 else -1.2),
                f'{v:+.1f}%', ha='center', fontsize=8, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(tier_labels, fontsize=11)
ax.set_ylabel('RMSE Reduction vs. Naïve (%)')
ax.set_title('ML Advantage Over Naïve by Article Age Tier\n(positive = ML wins)', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('plots/cold_start/CS2_ml_advantage_by_tier.png')
plt.close()

# ── Plot 3: RMSE vs product age (scatter per article) ─────────────────────────
# Compute per-article RMSE for best ML model and Naïve on test set
test_norm2 = normalise(test.copy())
lgbm = loaded_models.get('LightGBM (baseline)')

if lgbm is not None:
    feats = [f for f in lgbm.feature_name_ if f in test_norm2.columns]
    lgbm_pred = np.expm1(np.clip(lgbm.predict(test_norm2[feats]), 0, None))
    test_norm2['lgbm_pred'] = lgbm_pred

    art_metrics = []
    for art_id, grp in test_norm2.groupby('article_id'):
        y_t = grp['y_raw'].values
        y_p = grp['lgbm_pred'].values
        naive_p = df[(df['article_id']==art_id) &
                     (df['week_start'] >= pd.Timestamp('2020-01-06'))]['sales_lag1'].fillna(0).values
        age = article_age[article_age['article_id']==art_id]['age_at_test_start'].values
        if len(age) == 0: continue
        art_metrics.append({
            'article_id': art_id,
            'age': age[0],
            'lgbm_rmse':  rmse(y_t, y_p),
            'naive_rmse': rmse(y_t[:len(naive_p)], naive_p) if len(naive_p) else np.nan,
        })
    art_df = pd.DataFrame(art_metrics).dropna()
    art_df['delta_pct'] = (art_df['naive_rmse'] - art_df['lgbm_rmse']) / art_df['naive_rmse'] * 100

    fig, ax = plt.subplots(figsize=(11, 5))
    sc = ax.scatter(art_df['age'], art_df['delta_pct'], alpha=0.25, s=15,
                    c=art_df['delta_pct'], cmap='RdYlGn', vmin=-50, vmax=50)
    # Trend line
    z = np.polyfit(art_df['age'].clip(0,106), art_df['delta_pct'], 1)
    xline = np.linspace(0, 106, 200)
    ax.plot(xline, np.polyval(z, xline), color='navy', linewidth=2.5, label='Trend')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(4,  color='tomato',       linewidth=1.2, linestyle=':', label='Cold threshold (4w)')
    ax.axvline(16, color='darkorange',   linewidth=1.2, linestyle=':', label='Warm threshold (16w)')
    plt.colorbar(sc, ax=ax, label='Δ RMSE (LightGBM vs Naïve, %)')
    ax.set_xlabel('Article age at test start (weeks of history)')
    ax.set_ylabel('RMSE improvement over Naïve (%)')
    ax.set_title('LightGBM Advantage vs. Article Age\n(positive = ML better than Naïve)', fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('plots/cold_start/CS3_advantage_vs_age_scatter.png')
    plt.close()

print('Plots saved to plots/cold_start/')


Plots saved to plots/cold_start/


In [7]:
print('='*60)
print('COLD-START ANALYSIS SUMMARY')
print('='*60)
print()
for tier in TIERS:
    sub = results_df[results_df['tier'] == tier]
    naive_rmse = sub[sub['model']=='Naïve']['RMSE'].values
    best_ml_rmse = sub[sub['model']!='Naïve']['RMSE'].min()
    if len(naive_rmse):
        delta = (naive_rmse[0] - best_ml_rmse) / naive_rmse[0] * 100
        n = sub[sub['model']=='Naïve']['n_rows'].values[0]
        print(f'{tier:<22}  n={n:>6,}  Naïve RMSE={naive_rmse[0]:.3f}  '
              f'Best ML RMSE={best_ml_rmse:.3f}  ML advantage={delta:+.1f}%')

print()
print('Interpretation:')
print('  For cold-start articles (≤4 weeks of history), lag features are')
print('  near-zero and uninformative. The ML model falls back to product')
print('  attribute features (price, colour, index group), which partly')
print('  compensates — but the gap to established articles is significant.')
print()
print('Recommendation (future work):')
print('  For cold-start articles, use content-based features (product type,')
print('  colour, price point) as the primary signal. A two-stage model that')
print('  detects cold-start articles and applies a separate "launch model"')
print('  would likely reduce the cold-start RMSE gap.')


COLD-START ANALYSIS SUMMARY

Cold (≤4w)              n=81,548  Naïve RMSE=11456808035237710568426043452391256529956468641890866660076492137955328.000  Best ML RMSE=10.016  ML advantage=+100.0%
Warm (5–16w)            n= 9,576  Naïve RMSE=11008201914089707776665836433796194998226172489151905912100360550350848.000  Best ML RMSE=6.762  ML advantage=+100.0%
Established (>16w)      n=208,050  Naïve RMSE=6348193137299172319542405016423298151742175258336844357091576445403136.000  Best ML RMSE=3.743  ML advantage=+100.0%

Interpretation:
  For cold-start articles (≤4 weeks of history), lag features are
  near-zero and uninformative. The ML model falls back to product
  attribute features (price, colour, index group), which partly
  compensates — but the gap to established articles is significant.

Recommendation (future work):
  For cold-start articles, use content-based features (product type,
  colour, price point) as the primary signal. A two-stage model that
  detects cold-start articles a